# 00 · Mapillary Light Fetching — Islington
Fetches Mapillary street-light detections (`object--street-light`) via the Mapillary Graph API v4 for the London Borough of Islington, London, UK.
Output: one CSV saved to `data/mapillary/` with columns `id, lon, lat, first_seen_at, last_seen_at, object_type`.

## 0 · Imports & config

In [6]:
import os
import time
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import shape, Point

MAPILLARY_TOKEN_NAME = 'MAPILLARY_TOKEN'

def _parse_env_value(raw_value: str) -> str:
    value = raw_value.strip()
    if not value:
        return value

    if value[0] in {'"', '\''}:
        quote = value[0]
        end = 1
        while end < len(value):
            # Only accept an unescaped matching quote as the end
            if value[end] == quote and (end == 0 or value[end - 1] != '\\'):
                return value[1:end]
            end += 1
        return value[1:]

    if ' #' in value:
        value = value.split(' #', 1)[0].strip()
    elif '#' in value:
        value = value.split('#', 1)[0].strip()

    return value

def _read_env_file_value(env_path: Path, key: str) -> str | None:
    if not env_path.exists():
        return None

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        candidate_key, candidate_value = line.split('=', 1)
        if candidate_key.strip() != key:
            continue
        return _parse_env_value(candidate_value)

    return None

def _load_mapillary_token() -> str | None:
    token = os.environ.get(MAPILLARY_TOKEN_NAME)
    if token:
        return token

    for base_dir in (Path.cwd(), Path.cwd().parent):
        token = _read_env_file_value(base_dir / '.env', MAPILLARY_TOKEN_NAME)
        if token:
            return token

    return None

MAPILLARY_TOKEN = _load_mapillary_token()
# Sanitize token (strip, remove inline comments, drop surrounding quotes)
if MAPILLARY_TOKEN is not None:
    if '#' in MAPILLARY_TOKEN:
        MAPILLARY_TOKEN = MAPILLARY_TOKEN.split('#', 1)[0]
    MAPILLARY_TOKEN = MAPILLARY_TOKEN.strip()
    if MAPILLARY_TOKEN.startswith(('"', '\'')) and MAPILLARY_TOKEN.endswith(('"', '\'')):
        MAPILLARY_TOKEN = MAPILLARY_TOKEN[1:-1].strip()
# Masked token display for sanity (first/last 4 chars) -- safe to show
if MAPILLARY_TOKEN:
    t = MAPILLARY_TOKEN
    display = (t[:4] + '...' + t[-4:]) if len(t) > 8 else t
    print('MAPILLARY_TOKEN looks like:', display)

# ── Paths
DATA_DIR = Path('data/mapillary')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── CRS
CRS_GEO        = 'EPSG:4326'
CRS_PROJECTED  = 'EPSG:3857'

# ── API settings
BASE_URL       = 'https://graph.mapillary.com/map_features'
OBJECT_TYPE    = 'object--street-light'
PAGE_LIMIT     = 2000
SLEEP_BETWEEN  = 0.5


MAPILLARY_TOKEN looks like: MLY|...2f64


## 1 · Neighbourhood bounding box
Bounding box in `[min_lon, min_lat, max_lon, max_lat]` (EPSG:4326).
Tighten or expand if needed.

In [ ]:
NEIGHBOURHOODS = {
    'islington_london': [-0.1350, 51.4950, -0.0650, 51.5650],
}

print('Neighbourhoods configured:')
for name, bbox in NEIGHBOURHOODS.items():
    print(f'  {name:30s}  bbox={bbox}')

## 2 · Fetch helpers

In [7]:
def fetch_mapillary_lights_bbox(bbox: list[float], token: str) -> pd.DataFrame:
    """
    Fetch all Mapillary street-light map features inside *bbox*.
    Handles cursor-based pagination automatically.
"""
    records = []
    params = {
        'access_token': token,
        'fields':       'id,geometry,first_seen_at,last_seen_at,object_type',
        'object_type':  OBJECT_TYPE,
        'bbox':         ','.join(map(str, bbox)),
        'limit':        PAGE_LIMIT,
    }

    url = BASE_URL
    page = 0
    while url:
        resp = requests.get(url, params=params if page == 0 else None)
        resp.raise_for_status()
        payload = resp.json()

        for feat in payload.get('data', []):
            geom = feat.get('geometry', {})
            coords = geom.get('coordinates', [None, None])
            records.append({
                'id':            feat.get('id'),
                'lon':           coords[0],
                'lat':           coords[1],
                'first_seen_at': feat.get('first_seen_at'),
                'last_seen_at':  feat.get('last_seen_at'),
                'object_type':   feat.get('object_type'),
            })

        paging = payload.get('paging', {})
        next_url = paging.get('next')
        url = next_url
        params = None
        page += 1

        if next_url:
            time.sleep(SLEEP_BETWEEN)

    return pd.DataFrame(records)


def load_mapillary_lights(csv_path: Path) -> gpd.GeoDataFrame | None:
    if not csv_path.exists():
        print(f'  [WARNING] Mapillary CSV not found at {csv_path}.')
        print('  lamp_count will be set to NaN for all segments.')
        return None

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()

    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    if lat_col is None or lon_col is None:
        raise ValueError(f'Could not find lat/lon columns in {csv_path}. Found: {list(df.columns)}')

    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs=CRS_GEO).to_crs(CRS_PROJECTED)
    print(f'  Loaded {len(gdf):,} Mapillary light points from {csv_path.name}.' )
    return gdf


## 3 · Fetch & save neighbourhood

In [10]:
results = {}

def _split_bbox_into_tiles(bbox: list[float], grid_size: int = 3) -> list[list[float]]:
    min_lon, min_lat, max_lon, max_lat = bbox
    lon_step = (max_lon - min_lon) / grid_size
    lat_step = (max_lat - min_lat) / grid_size
    tiles = []
    for row in range(grid_size):
        for col in range(grid_size):
            tile_min_lon = min_lon + col * lon_step
            tile_max_lon = min_lon + (col + 1) * lon_step
            tile_min_lat = min_lat + row * lat_step
            tile_max_lat = min_lat + (row + 1) * lat_step
            tiles.append([tile_min_lon, tile_min_lat, tile_max_lon, tile_max_lat])
    return tiles

for name, bbox in NEIGHBOURHOODS.items():
    out_path = DATA_DIR / f'mapillary_lights_{name}.csv'

    if out_path.exists():
        print(f'[SKIP] {name} — already fetched ({out_path})')
        results[name] = pd.read_csv(out_path)
        continue

    print(f'[FETCH] {name} in tiles …')
    tile_frames = []
    for tile_index, tile_bbox in enumerate(_split_bbox_into_tiles(bbox, grid_size=3), start=1):
        print(f'  tile {tile_index}/9: {tile_bbox}', flush=True)
        try:
            tile_df = fetch_mapillary_lights_bbox(tile_bbox, MAPILLARY_TOKEN)
            if not tile_df.empty:
                tile_frames.append(tile_df)
                print(f'    {len(tile_df):,} lights')
            else:
                print('    0 lights')
        except requests.HTTPError as e:
            print(f'    ERROR: {e}')
        time.sleep(SLEEP_BETWEEN)

    if tile_frames:
        df = pd.concat(tile_frames, ignore_index=True)
        if 'id' in df.columns:
            df = df.drop_duplicates(subset=['id']).reset_index(drop=True)
        df.to_csv(out_path, index=False)
        results[name] = df
        print(f'[DONE] {len(df):,} lights → {out_path}')
    else:
        results[name] = None
        print(f'[DONE] No lights fetched for {name}')

[FETCH] islington_london in tiles …
  tile 1/9: [-0.135, 51.495, -0.11166666666666668, 51.51833333333333]
    1,803 lights
  tile 2/9: [-0.11166666666666668, 51.495, -0.08833333333333335, 51.51833333333333]
    1,655 lights
  tile 3/9: [-0.08833333333333335, 51.495, -0.065, 51.51833333333333]
    1,674 lights
  tile 4/9: [-0.135, 51.51833333333333, -0.11166666666666668, 51.541666666666664]
    1,790 lights
  tile 5/9: [-0.11166666666666668, 51.51833333333333, -0.08833333333333335, 51.541666666666664]
    1,799 lights
  tile 6/9: [-0.08833333333333335, 51.51833333333333, -0.065, 51.541666666666664]
    1,941 lights
  tile 7/9: [-0.135, 51.541666666666664, -0.11166666666666668, 51.565]
    1,583 lights
  tile 8/9: [-0.11166666666666668, 51.541666666666664, -0.08833333333333335, 51.565]
    1,797 lights
  tile 9/9: [-0.08833333333333335, 51.541666666666664, -0.065, 51.565]
    1,880 lights
[DONE] 15,922 lights → data\mapillary\mapillary_lights_islington_london.csv


## 4 · Summary

In [11]:
summary = [
    {
        'neighbourhood': name,
        'n_lights':      len(df) if df is not None else 'ERROR',
        'csv':           str(DATA_DIR / f'mapillary_lights_{name}.csv'),
    }
    for name, df in results.items()
]
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

   neighbourhood  n_lights                                                  csv
islington_london     15922 data\mapillary\mapillary_lights_islington_london.csv


## 5 · Quick sanity check — load back as GeoDataFrames

In [12]:
gdfs = {}
for name in NEIGHBOURHOODS:
    csv_path = DATA_DIR / f'mapillary_lights_{name}.csv'
    print(f'Loading {name}:')
    gdfs[name] = load_mapillary_lights(csv_path)


Loading islington_london:
  Loaded 15,922 Mapillary light points from mapillary_lights_islington_london.csv.


## 6 · Quick visual check (optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(gdfs), figsize=(5 * max(1, len(gdfs)), 5))
if len(gdfs) == 1:
    axes = [axes]

for ax, (name, gdf) in zip(axes, gdfs.items()):
    if gdf is None or gdf.empty:
        ax.set_title(f'{name}
(no data)')
        continue
    gdf.plot(ax=ax, markersize=2, color='orange', alpha=0.6)
    ax.set_title(f'{name}
({len(gdf):,} lights)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR / 'mapillary_lights_overview.png', dpi=150)
plt.show()
print('Saved overview plot to', DATA_DIR / 'mapillary_lights_overview.png')